In [1]:
# Imports
import os
import torch
import time
import wikipediaapi
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM
from transformers import AutoTokenizer

In [2]:
# Global Settings
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Configure Chunking
Settings.chunk_size = 512
Settings.chunk_overlap = 50

# Configure LLM
Settings.llm = HuggingFaceLLM(
    model_name=model_id,
    tokenizer_name=model_id,
    context_window=2048,
    max_new_tokens=256,
    generate_kwargs={"temperature": 0.7, "do_sample": True},
    device_map="auto", 
    model_kwargs={"dtype": torch.float16 if torch.cuda.is_available() else torch.float32}
)

# Configure Embeddings
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

print("Local models and settings initialized.")

Local models and settings initialized.


In [3]:
# Fetch Wikipedia Data
user_agent = "GenAI_Project_Bot/1.0 (academic_research_contact@example.com)"
wiki = wikipediaapi.Wikipedia(user_agent, 'en')

pages = [
    "Generative artificial intelligence",
    "Large language model",
    "Transformer (machine learning)",
    "Diffusion model"
]

if not os.path.exists("./data"):
    os.makedirs("./data")

for page_title in pages:
    page = wiki.page(page_title)
    if page.exists():
        file_name = page_title.replace(" ", "_").replace("/", "-") + ".txt"
        with open(os.path.join("./data", file_name), "w", encoding="utf-8") as f:
            f.write(page.text)
        print(f"✅ Saved: {page_title}")

print("\nData download complete.")

✅ Saved: Generative artificial intelligence
✅ Saved: Large language model
✅ Saved: Transformer (machine learning)
✅ Saved: Diffusion model

Data download complete.


In [4]:
# Build Index
documents = SimpleDirectoryReader("./data").load_data()
index = VectorStoreIndex.from_documents(
    documents, 
    transformations=[SentenceSplitter(chunk_size=512, chunk_overlap=50)]
)

query_engine = index.as_query_engine(similarity_top_k=2)
print(f"Index built successfully with {len(documents)} documents.")

Index built successfully with 6 documents.


In [5]:
# Cell 5: Run Evaluation
test_questions = [
    "What is the core idea of a Transformer architecture?",
    "How do Large Language Models (LLMs) differ from traditional AI?",
    "What role does the 'diffusion' process play in image generation?"
]

for i, question in enumerate(test_questions, 1):
    start_time = time.time()
    response = query_engine.query(question)
    duration = time.time() - start_time
    
    print(f"TEST {i}: {question}")
    print(f"⏱️ Time: {duration:.2f}s | 📝 RESPONSE: {str(response).strip()}\n")

TEST 1: What is the core idea of a Transformer architecture?
⏱️ Time: 25.50s | 📝 RESPONSE: The core idea of a Transformer architecture is to replace the traditional sequential processing of RNNs with attention mechanisms in order to overcome the vanishing gradient problem in long-sequence modelling. The transformer architecture has since become a standard for long-sequence modelling, specifically using the multiplicative units used by LSTMs to tackle the vanishing gradient problem. The transformer is a family of artificial neural network architectures based on the multi-head attention mechanism, and has achieved success in various applications including speech-to-text, machine translation, computer vision, and natural language processing, among others.

TEST 2: How do Large Language Models (LLMs) differ from traditional AI?
⏱️ Time: 28.60s | 📝 RESPONSE: Large Language Models (LLMs) are a computational model based on transformer architectures that can generate, summarize, translate, and